# Evidence: Excel for the AI

**What it does**
Converts Excel evidence (`.xlsx`, `.xlsm`) in the evidences folder into files the AI can read, and shows the result:
- `<name>.xlsx.md`: every sheet as a table with the Excel column letters and row numbers, so the AI can cite a cell like "F14". Colors, comments, merged cells, hidden rows, columns and sheets, and formulas are written next to the values, for example `{row fill: light red}` or `{comment: "..."}`.
- `<name>.xlsx_images/`: pictures pasted into the sheets (screenshots), sent to the AI as images. The text says where each one sits.
- `<name>.xlsx.pdf`: a visual copy of every sheet, only when LibreOffice is installed.

**Why**
Gemini and Claude do not accept Excel files. A PDF alone is poor for wide sheets (10 to 40 columns): tiny text, columns split across pages, and models are not precise at reading values off a page. The conversion lives in the `ExcelConverter` service (`services/evidences/excel_converter.py`), so the API can use the same code later. Plain CSV keeps the values but loses colors (red = exception), comments and pictures. Text with cell addresses for the exact values, plus images for what can only be seen, is the recommended approach (README, "Excel evidence").

**How to use**
1. Put Excel files in the evidences folder (`storage/evidences/` or `KCT_EVIDENCES_ROOT`). Subfolders are fine. No file yet? Run Step 1 to create a fake ING-style workbook.
2. Run all cells. To convert one file, put its name in `FILE` (Step 3).
3. Read the text in Step 4: what you see there is exactly what the AI gets.

**Not covered without LibreOffice**: charts, colors from conditional formatting (flagged in the text), and formulas in a file never saved by Excel (shown as the formula).

**Needs**: `openpyxl` (`pip install openpyxl`, or Step 3 of `01_setup.ipynb`).

## Setup

Finds the project folder, reads `.env`, and shows the evidences folder and whether LibreOffice is there for the PDF copy.

In [ ]:
import os, sys
from pathlib import Path

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from services.app import boot
from services.evidences import libreoffice

kct = boot()
evidences = kct.evidences

print("Evidences folder:", evidences.root)
print("LibreOffice:     ", libreoffice.binary() or "not installed: text and images only, no PDF copy")

## Step 1: Fake ING-style workbook (optional)

Creates `EV-03_ADX_audit_log_extract_SAMPLE.xlsx` with `ExcelFactory` (`factories/excel_factory.py`): made-up data and what real evidence workbooks contain:
- 4 sheets, one of them hidden
- a 20-column table with 100 rows, with a merged title, a colored header and light red exception rows with red text
- reviewer comments, a hidden column and a hidden row
- a two-level header (group headers merged over their columns)
- a form-style summary with formulas, conditional formatting and a pasted screenshot

`ROWS` sets how many event rows sheet 1 gets. Running it again overwrites the file.

In [ ]:
from factories import ExcelFactory

ROWS = 100
print("Created", ExcelFactory(rows=ROWS).create(evidences.root))

## Step 2: Excel files in the evidences folder

Every workbook, with its size. "needs conversion" means there is no `.md` yet, or the workbook changed after it was made.

In [ ]:
workbooks = [p for p in evidences.files() if p.suffix.lower() in (".xlsx", ".xlsm") and not p.name.startswith("~$")]
if not workbooks:
    print("No Excel files in", evidences.root, "- add some or run Step 1")
for relative in workbooks:
    path = evidences.path(relative)
    text = path.with_name(path.name + ".md")
    state = "converted" if text.is_file() and text.stat().st_mtime >= path.stat().st_mtime else "needs conversion"
    print(f"{str(relative):<60} {path.stat().st_size / 1024:>8.1f} KB  {state}")

## Step 3: Convert one workbook

Set `FILE` to a name from the list above (relative to the evidences folder). Prints each sheet's size, the pictures found, and an estimate of the tokens the text costs when sent to the AI.

In [ ]:
FILE = "EV-03_ADX_audit_log_extract_SAMPLE.xlsx"

try:
    result = evidences.convert_excel(FILE)
except Exception as exc:
    print("❌", type(exc).__name__, exc)
else:
    print(f"✅ {result.text.name}  (about {result.estimated_tokens:,} tokens, engine: {result.engine})")
    for sheet in result.sheets:
        flags = " hidden" if sheet.hidden else ""
        print(f"   sheet {sheet.name!r}: {sheet.rows} rows x {sheet.columns} columns, {sheet.images} picture(s){flags}")
    for image in result.images:
        print("   picture:", image.relative_to(evidences.root))
    if result.pdf:
        print("   visual copy:", result.pdf.name)

## Step 4: What the AI will read

The start of the text made in Step 3. Look for `{row fill: ...}`, `{comment: ...}`, `(hidden)` and `{image ...}`. Raise `LINES` to see more; the full file is next to the workbook.

In [ ]:
LINES = 40

lines = result.text.read_text(encoding="utf-8").splitlines()
for line in lines[:LINES]:
    print(line[:250] + (" …" if len(line) > 250 else ""))
print(f"\n… {len(lines)} lines in total: {result.text}")

## Step 5: Show the pictures

The pictures taken out of the workbook, as the AI will see them.

In [ ]:
from IPython.display import Image, display

for image in result.images:
    print(image.name)
    display(Image(filename=str(image)))

## Step 6: Convert everything that needs it

Converts every workbook in the folder (and subfolders) whose `.md` is missing or older than the workbook. `FORCE = True` converts all of them again.

In [ ]:
FORCE = False

converted = evidences.convert_all_excel(force=FORCE)
if not converted:
    print("✅ Every workbook is already converted")
for item in converted:
    print("✅", item.text.relative_to(evidences.root), f"(about {item.estimated_tokens:,} tokens, {len(item.images)} picture(s))")

## Step 7: Ready for the AI

Every file in the evidences folder the AI can read (PDFs from Word, workbook texts, pictures) as attachments. The assessment will send these with the test steps.

In [ ]:
for doc in evidences.attachments():
    print(f"{doc.title:<75} {len(doc.data) / 1024:>8.1f} KB  {doc.mime_type}")